In [68]:
import requests
import json
from datetime import datetime
from datetime import timezone
import time
import random
import asyncio, aiohttp

_TOKEN_CACHE = None  # global cache

def get_session_token():
    global _TOKEN_CACHE
    if _TOKEN_CACHE:   # reuse valid token
        return _TOKEN_CACHE

    url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    handle = "repostproj.bsky.social"
    app_password = "vyvc-xg5q-seda-utaz" 

    for _ in range(3):  # retry a few times on rate limit
        r = requests.post(url, json={"identifier": handle, "password": app_password})
        if r.status_code == 429:
            print("⚠️ Rate-limited on login, sleeping 5s...")
            time.sleep(1)
            continue
        r.raise_for_status()
        _TOKEN_CACHE = r.json()["accessJwt"]
        return _TOKEN_CACHE

    raise RuntimeError("Failed to obtain token after retries")


In [69]:


def search_bluesky_posts(
    query,
    limit=1000,
    outfile="posts.jsonl",
    since=None,
    until=None,
    lang="en",
    sort="latest",
):
    """
    Search Bluesky posts using the authenticated app.bsky.feed.searchPosts endpoint.
    Supports since/until, lang, sort.
    """
    url = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
    token = get_session_token()
    headers = {
        "Authorization": f"Bearer {token}",
        "User-Agent": "Mozilla/5.0",
        "Accept-Encoding": "gzip",
    }

    cursor, total = None, 0

    def fmt(dt):
        if dt is None:
            return None
        if isinstance(dt, datetime):
            # Use Zulu (UTC) format that Bluesky expects
            return dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        return str(dt)

    since = fmt(since)
    until = fmt(until)

    with open(outfile, "a", encoding="utf-8") as f:
        while total < limit:
            params = {
                "q": query,
                "sort": sort,
                "lang": lang,
                "limit": min(100, limit - total),
            }
            if cursor:
                params["cursor"] = cursor
            if since:
                params["since"] = since
            if until:
                params["until"] = until

            r = requests.get(url, params=params, headers=headers)
            if r.status_code in (429, 502):
                print(f"⚠️ Rate limited ({r.status_code}) — waiting 1s...")
                time.sleep(1)
                continue
            if r.status_code == 401:
                raise RuntimeError("Invalid or expired token — get a new session token.")
            if r.status_code != 200:
                raise RuntimeError(f"Error {r.status_code}: {r.text}")

            data = r.json()
            posts = data.get("posts", [])
            if not posts:
                print("ℹNo posts in this batch — stopping.")
                break

            for p in posts:
                json.dump(p, f, ensure_ascii=False)
                f.write("\n")
                total += 1
                if total >= limit:
                    break

            cursor = data.get("cursor")
            if not cursor:
                print("ℹNo more cursor — reached end of results.")
                break

            print(f"🧾 Collected {total} posts so far...")

    print(f"✅ Saved {total} posts to {outfile}")


In [93]:


search_bluesky_posts(
    query="#WWERaw",
    limit=3000,
    outfile="breachdallas.jsonl",
    until=datetime(2025, 10, 21)
)

🧾 Collected 100 posts so far...
🧾 Collected 200 posts so far...
🧾 Collected 299 posts so far...
🧾 Collected 399 posts so far...
🧾 Collected 499 posts so far...
🧾 Collected 598 posts so far...
🧾 Collected 698 posts so far...
🧾 Collected 798 posts so far...
🧾 Collected 898 posts so far...
🧾 Collected 997 posts so far...
🧾 Collected 1097 posts so far...
🧾 Collected 1197 posts so far...
🧾 Collected 1297 posts so far...
🧾 Collected 1397 posts so far...
🧾 Collected 1497 posts so far...
🧾 Collected 1597 posts so far...
🧾 Collected 1697 posts so far...
🧾 Collected 1797 posts so far...
🧾 Collected 1897 posts so far...
🧾 Collected 1997 posts so far...
🧾 Collected 2097 posts so far...
🧾 Collected 2197 posts so far...
🧾 Collected 2297 posts so far...
🧾 Collected 2397 posts so far...
🧾 Collected 2497 posts so far...
🧾 Collected 2596 posts so far...
🧾 Collected 2695 posts so far...
🧾 Collected 2795 posts so far...
🧾 Collected 2895 posts so far...
🧾 Collected 2995 posts so far...
🧾 Collected 3000 pos

In [100]:

def count_trump_tagged_posts(tag, file_path="breachdallas.jsonl"):
    """
    Counts how many Bluesky posts in a JSONL file include 'trump' as a tag.
    
    Args:
        file_path (str): Path to the JSONL file containing Bluesky post objects.
    
    Returns:
        int: Number of posts that have 'trump' as a tag.
    """
    count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                post = json.loads(line.strip())
                record = post.get('record', {})
                facets = record.get('facets', [])
                
                # Extract tags from facets
                tags = []
                for facet in facets:
                    for feature in facet.get('features', []):
                        if feature.get('$type') == 'app.bsky.richtext.facet#tag':
                            tags.append(feature.get('tag', '').lower())

                if tag.lower() in tags:
                    count += 1

            except json.JSONDecodeError:
                continue  # skip malformed l
        return count

print(count_trump_tagged_posts('WWERaw'))


3000


In [101]:
import json
from datetime import datetime
from collections import Counter

def get_post_days(file_path="breachdallas.jsonl", with_counts=False):
    dates = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                p = json.loads(line)
            except json.JSONDecodeError:
                continue

            created = (p.get("record") or {}).get("createdAt")
            if not created:
                continue

            # Safe ISO parsing (handles trailing 'Z')
            try:
                d = datetime.fromisoformat(created.replace("Z", "+00:00")).date().isoformat()
            except Exception:
                d = created[:10]  # fallback

            dates.append(d)

    if with_counts:
        return dict(sorted(Counter(dates).items()))  # {date: num_posts}
    return sorted(set(dates))  # ["YYYY-MM-DD", ...]
get_post_days()

['2025-09-16',
 '2025-09-17',
 '2025-09-18',
 '2025-09-19',
 '2025-09-20',
 '2025-09-21',
 '2025-09-22',
 '2025-09-23',
 '2025-09-24',
 '2025-09-25',
 '2025-09-26',
 '2025-09-27',
 '2025-09-28',
 '2025-09-29',
 '2025-09-30',
 '2025-10-01',
 '2025-10-02',
 '2025-10-03',
 '2025-10-04',
 '2025-10-05',
 '2025-10-06',
 '2025-10-07',
 '2025-10-08',
 '2025-10-09',
 '2025-10-10',
 '2025-10-11',
 '2025-10-12',
 '2025-10-13',
 '2025-10-14',
 '2025-10-15',
 '2025-10-16',
 '2025-10-17',
 '2025-10-18',
 '2025-10-19',
 '2025-10-20']

In [102]:
import requests
import json
import time

API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"

def fetch_reposters(uri, limit=100):
    """Fetch reposters for one post URI (synchronous)."""
    params = {"uri": uri, "limit": limit}
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r = requests.get(API_URL, params=params, headers=headers, timeout=60)
        if r.status_code != 200:
            print(f"⚠️ HTTP {r.status_code} for {uri}")
            return []
        data = r.json()
        return [u["did"] for u in data.get("repostedBy", [])]
    except Exception as e:
        print(f"⚠️ Error for {uri}: {e}")
        return []


def collect_all_reposters_sync(file_path):
    """Fetch reposters for all posts sequentially (no async)."""
    results = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            post = json.loads(line)
            uri = post.get("uri") or post.get("post", {}).get("uri")
            if not uri:
                continue

            print(f"🔍 Fetching reposters for post {i+1}: {uri}")
            reposters = fetch_reposters(uri)
            if not reposters:
                continue

            post_id = uri or f"post_{i}"
            results[post_id] = {"post": post, "reposters": reposters}


    print(f"✅ Done. Collected reposters for {len(results)} posts.")
    return results


In [103]:
reposts = collect_all_reposters_sync("breachdallas.jsonl")


🔍 Fetching reposters for post 1: at://did:plc:uqpjammmeuctpl6mxfflotwu/app.bsky.feed.post/3m3nzkf3ftc2t
🔍 Fetching reposters for post 2: at://did:plc:3jod2mt7f7jva63p5wagydbv/app.bsky.feed.post/3m3nzkb7b6c2v
🔍 Fetching reposters for post 3: at://did:plc:sciua6fe75aeejnrkxq5tbng/app.bsky.feed.post/3m3nzjogwac2o
🔍 Fetching reposters for post 4: at://did:plc:2nyvnhpggbjqold2tbnvzilo/app.bsky.feed.post/3m3nzbix5ok2p
🔍 Fetching reposters for post 5: at://did:plc:nc4lnxtbijqsda7pjj234ml2/app.bsky.feed.post/3m3nz2t7w7k22
🔍 Fetching reposters for post 6: at://did:plc:bpd5vxxuk3mfg5t7xn2hx2jn/app.bsky.feed.post/3m3nyrxqmxk2g
🔍 Fetching reposters for post 7: at://did:plc:ohryblxb3umxan3bkxzsux74/app.bsky.feed.post/3m3ny5o7prk25
🔍 Fetching reposters for post 8: at://did:plc:5biaq4juulxpglvm6tf4c7si/app.bsky.feed.post/3m3nx5fqdz22w
🔍 Fetching reposters for post 9: at://did:plc:h2dqdi7zi3n6bazjqmpxbx3h/app.bsky.feed.post/3m3nwhwwou22u
🔍 Fetching reposters for post 10: at://did:plc:2f5xrk2cvjsds5vu4

KeyboardInterrupt: 